Model to predict Cancellation of the hotel booking

Importing libraries

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib as plt
import seaborn as sns

Loading the dataset

In [100]:
df = pd.read_csv("H1.csv")
df.head(20)

,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
0,0,342,2015,July,27,1,0,0,2,0,...,No Deposit,NULL,NULL,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,0,737,2015,July,27,1,0,0,2,0,...,No Deposit,NULL,NULL,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,0,7,2015,July,27,1,0,1,1,0,...,No Deposit,NULL,NULL,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,0,13,2015,July,27,1,0,1,1,0,...,No Deposit,304,NULL,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,0,14,2015,July,27,1,0,2,2,0,...,No Deposit,240,NULL,0,Transient,98.00,0,1,Check-Out,2015-07-03
5,0,14,2015,July,27,1,0,2,2,0,...,No Deposit,240,NULL,0,Transient,98.00,0,1,Check-Out,2015-07-03
6,0,0,2015,July,27,1,0,2,2,0,...,No Deposit,NULL,NULL,0,Transient,107.00,0,0,Check-Out,2015-07-03
7,0,9,2015,July,27,1,0,2,2,0,...,No Deposit,303,NULL,0,Transient,103.00,0,1,Check-Out,2015-07-03
8,1,85,2015,July,27,1,0,3,2,0,...,No Deposit,240,NULL,0,Transient,82.00,0,1,Canceled,2015-05-06
9,1,75,2015,July,27,1,0,3,2,0,...,No Deposit,15,NULL,0,Transient,105.50,0,0,Canceled,2015-04-22


Shape of dataset

In [ ]:
df.shape

Datatype 

In [ ]:
df.dtypes

Total memory usage

In [ ]:
df.memory_usage(deep=True).sum()

Total Duplicated rows

In [ ]:
df.duplicated().sum()

In [101]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

Checking for missing values

In [ ]:
df.isnull().sum()

In [102]:
df['Country'].fillna('Unknown', inplace=True)

C:\Users\MSII\AppData\Local\Temp\ipykernel_9736\1255347460.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Country'].fillna('Unknown', inplace=True)


In [ ]:
df.isnull().sum()

In [103]:
month_map = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}
df['ArrivalDateMonth'] = df['ArrivalDateMonth'].map(month_map)

In [ ]:
df.nunique()

Dropping columns

To prevent data leakage-> ReservationStatus, ReservationStatusDate

Agent and Company have many "NULL" values-> we can safely remove them

AssignedRoomType-> It can change after booking

ArrivalDateWeekNumber-> Month is already being used



In [104]:
df = df.drop(columns=["ReservationStatus", "ReservationStatusDate", "Agent", "Company", "AssignedRoomType", "ArrivalDateWeekNumber"])
df.shape

(33968, 25)

Feature engineering

In [105]:
df["is_peak_season"] = df['ArrivalDateMonth'].isin([6,7,8,12]).astype(int)  

In [106]:
df["Total_guests"] = df['Adults'] + df["Babies"]+ df['Children']

In [107]:
df["isSoloTraveler"] = (df['Total_guests'] == 1).astype(int)

In [108]:
df['HasChildren'] = (df['Children']>0).astype(int)

In [109]:
df['TotalStay'] = (df['StaysInWeekendNights'] - df["StaysInWeekNights"]).astype(int)

In [110]:
df['IsLeadTimeHigh'] = (df['LeadTime']>90).astype(int)

In [111]:
df["HadPreviousCancellaion"] = (df['PreviousCancellations']>0).astype(int)

In [112]:
df['isADRHigh'] = (df['ADR'] >df['ADR'].median()).astype(int)

In [117]:
df = df.drop(columns=['Adults', 'Children', 'Babies'])

Encoding

In [ ]:
df =  pd.get_dummies(df, columns =["CustomerType", "DepositType", "DistributionChannel", "Meal", "MarketSegment"],drop_first=True)

In [119]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [115]:
df = pd.get_dummies(df, columns=['Country', 'ReservedRoomType'], drop_first=True)

In [120]:
df.dtypes.value_counts()

int64      173
float64      1
Name: count, dtype: int64